**Encoder** - Day 4 Code

In [2]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Encoder, self).__init__()

        # Word -> Vector
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # LSTM processes the sequence 
        self.lstm = nn.LSTM(embed_size, 
                            hidden_size, 
                            #num_layers=2,
                            #dropout=0.3,
                            #bidirectional=True,
                            batch_first=True)

    def forward(self, x):
        # x shape:[batch, seq_len]
        embedded = self.embedding(x)
        # embedded shape: [batch, seq_len, embed_size]

        output, (hidden, cell) = self.lstm(embedded)

        # hidden, cell -> context vector passed to encoder 

        return hidden, cell

# _______ Test Encoder ____________________________________
encoder = Encoder(vocab_size=100, embed_size=16, hidden_size=32)

# Simulate input: batch=1, seq_len=5( % Words)
src = torch.tensor([[4, 23, 67, 12, 8]])

hidden, cell = encoder(src)
print("Encoder hidden shape:", hidden.shape)   
print("Encoder cell shape:", cell.shape)

Encoder hidden shape: torch.Size([1, 1, 32])
Encoder cell shape: torch.Size([1, 1, 32])


**Decoder** 

In [7]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Decoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

        # Final layer : hidden + word prediction
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x , hidden, cell):
        # x shape: [batch] -> add sqe dimention
        x = x.unsqueeze(1)
        
        embedded = self.embedding(x)
        # embedded [batch, 1 , embed_size]
        
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: [batch, 1, hidden_size]
        
        prediction = self.fc(output.squeeze(1))
        # predict: [batch, vocab_size]
        
        return prediction, hidden, cell

# ________________ Test Decoder _____________________
decoder = Decoder(vocab_size=100, embed_size=16,hidden_size=32)

start_token = torch.tensor([1])
# Start token (like <sos> = start of sentence

pred, hidden, cell = decoder(start_token, hidden, cell)
print("Decoder Prediction Shape:", pred.shape)
print("Predict word next:", pred.argmax().item())

Decoder Prediction Shape: torch.Size([1, 100])
Predict word next: 15


**Full Seq2Seq** — Encoder + Decoder Together

In [8]:
class Sqe2Sqe(nn.Module):
    def __init__(self, encoder, decoder):
        super(Sqe2Sqe, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, target_len):
        # Step 1: encode input
        hidden, cell = self.encoder(src)

        # Step 2: Start decode with <sos> token (index=1)
        batch_size = src.shape[0]
        x = torch.ones(batch_size, dtype=torch.long) # <SOS> token

        outputs = []

        # Step 3: Decode word to word
        for _ in range(target_len):
            pred, hidden, cell = self.decoder(x, hidden, cell)
            outputs.append(pred)

            # Next input = predicted word
            x = pred.argmax(dim=1)
            
        return torch.stack(outputs, dim=1)
        # Shape:[batch, target_len, vocal-size)
# ________________Test Full Sqe2Sqe_____________________________
encoder = Encoder(vocab_size=100, embed_size=16, hidden_size=32)
decoder = Decoder(vocab_size=100, embed_size=16, hidden_size=32)
model = Sqe2Sqe(encoder,decoder)

src = torch.tensor([[4,23,67,12,8]])   # Input sentence
out = model(src, target_len=4)

print("Sqe2Sqe output shape:", out.shape)  
print("Predicted word indices:", out.argmax(dim=2))

Sqe2Sqe output shape: torch.Size([1, 4, 100])
Predicted word indices: tensor([[31, 38, 81, 57]])
